# Eval + Inference — Gemma LoRA VLM (E2B / E4B)

Đánh giá 2 model **Gemma 3n VLM fine-tune LoRA** trên **VQA-RAD** + **RadImageNet-VQA**, và xuất **50 inference CSV** mỗi model.

| Model | Repo LoRA | Base (unsloth) |
|---|---|---|
| gemma_E4B | `VQA-DeepLearning/gemma_4_lora_E4b` | `unsloth/gemma-4-E4B-it` |
| gemma_E2B | `VQA-DeepLearning/gemma_2_lora_E2b` | `unsloth/gemma-4-E2B-it` |

**Load cái gì:** base Gemma + LoRA adapter từ **`checkpoint-400`** (bản tốt nhất đã chốt) — đổi ở `CONFIG` nếu muốn checkpoint khác.

**Đủ cột để so với A1/A2...:** metrics dùng chung `score_predictions` (em/f1/bleu/clin_*/sem_open) + thêm `params_total`/`params_trainable`/`inference_time_sec` → **trùng bộ cột** với các run CLIP+GPT2 nên gộp/so bảng được. *(Gemma sinh tự do, không có `p_yes` softmax nên `clin_*` suy từ text — cột vẫn đủ, chỉ AUC kém nghĩa hơn.)*

> Inference VLM chậm (~1-3s/mẫu). Full 2 test (~951 mẫu) × 2 model có thể ~1h. Muốn nhanh: đặt `EVAL_LIMIT` = 1 số (vd 150)."

## BƯỚC 1: Setup (code repo + unsloth + HF login)

In [ ]:
# 1.1 Lay code repo (de dung src/metrics + file parquet RadImageNet). DAN token GitHub.
GITHUB_TOKEN = ''
GITHUB_USER  = 'HoangVuSnape'; REPO = '02_Code_DeepLearning'
import os
if os.path.isdir(REPO + '/.git'):
    get_ipython().system('cd ' + REPO + ' && git fetch origin && git reset --hard origin/main && git log --oneline -1')
elif not os.path.isdir('.git'):
    get_ipython().system('git clone https://' + GITHUB_USER + ':' + GITHUB_TOKEN + '@github.com/' + GITHUB_USER + '/' + REPO + '.git')
if os.path.isdir('02_Code_DeepLearning'): os.chdir('02_Code_DeepLearning')
print('CWD:', os.getcwd())

In [ ]:
%%capture
# 1.2 Cai unsloth (y het notebook train Gemma) + thu vien cham diem
get_ipython().system('pip install unsloth')
get_ipython().system('pip install -q sentence-transformers scikit-learn python-dotenv hf_transfer')

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# KHONG dat PYTORCH_CUDA_ALLOC_CONF=expandable_segments (xung khac bitsandbytes 4bit -> illegal memory access)
from dotenv import load_dotenv
load_dotenv('.env') if os.path.exists('.env') else load_dotenv()
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
from huggingface_hub import login
if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN']); print('HF login OK')
else:
    print('Thieu HF_TOKEN - se khong push duoc')

## BƯỚC 2: CONFIG

In [ ]:
MODELS = [
    {'name': 'gemma_E4B', 'repo': 'VQA-DeepLearning/gemma_4_lora_E4b', 'checkpoint': 'checkpoint-400'},
    {'name': 'gemma_E2B', 'repo': 'VQA-DeepLearning/gemma_2_lora_E2b', 'checkpoint': 'checkpoint-400'},
]
# checkpoint-400 = ban tot nhat (ban da chot). Doi so neu muon checkpoint khac; =None -> adapter o ROOT repo.

PARQUET     = 'radimagenet_vqa_audit/radimagenet_vqa_500_test.parquet'
EVAL_LIMIT  = None    # None = full test; hoac dat 1 so (vd 150) cho nhanh
N_INFER     = 50      # so mau xuat inference CSV
PUSH_HF     = True    # push metrics + csv len chinh repo cua tung model (subfolder eval/)
MAX_NEW_TOK = 16
print('Models:', [(m['name'], m['checkpoint']) for m in MODELS], '| EVAL_LIMIT:', EVAL_LIMIT)

## BƯỚC 3: Helper — load Gemma LoRA + hàm inference (bám code train)

In [ ]:
import torch
from unsloth import FastVisionModel, get_chat_template
from huggingface_hub import snapshot_download

def load_gemma(m):
    if m.get('checkpoint'):
        snapshot_download(repo_id=m['repo'], allow_patterns=m['checkpoint'] + '/*',
                          local_dir='gemma_dl/' + m['name'])
        path = 'gemma_dl/' + m['name'] + '/' + m['checkpoint']
    else:
        path = m['repo']
    print('Load', m['name'], 'tu', path, '...', flush=True)
    
    # Giải phóng bộ nhớ GPU trước khi load
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    if m['name'] == 'gemma_E4B':
        # LOADER 2-GPU cho E4B sử dụng native transformers + PEFT
        # Trải mô hình lên cả 2 GPU T4 của Kaggle (device_map='auto') để tránh OOM
        print("-> Đang sử dụng Loader 2-GPU (transformers + PEFT) cho E4B...")
        from transformers import AutoProcessor, BitsAndBytesConfig
        from peft import PeftModel
        
        # Nhập class AutoModel phù hợp một cách robust
        model_class = None
        for class_name in ["AutoModelForImageTextToText", "AutoModelForVision2Seq", "Gemma4ForConditionalGeneration", "AutoModel"]:
            try:
                mod = __import__("transformers", fromlist=[class_name])
                model_class = getattr(mod, class_name)
                print(f"-> Đã import thành công {class_name} từ transformers.")
                break
            except (ImportError, AttributeError):
                continue
        
        if model_class is None:
            raise ImportError("Không tìm thấy class phù hợp (AutoModelForImageTextToText/AutoModelForVision2Seq/AutoModel) trong thư viện transformers.")
            
        base_model_name = "unsloth/gemma-4-E4B-it"
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        
        # Load base model tự động chia sẻ trên 2 GPU
        base_model = model_class.from_pretrained(
            base_model_name,
            quantization_config=bnb_config,
            dtype=torch.float16,
            device_map="auto"
        )
        
        # Load LoRA adapter weights
        model = PeftModel.from_pretrained(base_model, path)
        processor = AutoProcessor.from_pretrained(base_model_name)
        
        # Áp dụng template chat cho tokenizer của processor (vì processor native không có padding_side)
        processor.tokenizer = get_chat_template(processor.tokenizer, chat_template='gemma-4')
        model.eval()
    else:
        # Sử dụng Unsloth Loader thông thường cho E2B vì nhẹ và fit được 1 GPU
        print("-> Đang sử dụng Unsloth Loader cho E2B...")
        kw = dict(
            load_in_4bit=True, 
            dtype=torch.float16, 
            float32_mixed_precision=False
        )
        try:
            model, processor = FastVisionModel.from_pretrained(path, **kw)
        except TypeError:
            model, processor = FastVisionModel.from_pretrained(path, load_in_4bit=True, dtype=torch.float16)
        FastVisionModel.for_inference(model)
        processor = get_chat_template(processor, chat_template='gemma-4')
        
    return model, processor

def gemma_predict(model, processor, image, question):
    # KHOP DUNG FORMAT LUC TRAIN: text (cau hoi THO, khong them prompt) TRUOC, image SAU
    messages = [{'role': 'user', 'content': [
        {'type': 'text', 'text': str(question)},
        {'type': 'image'}]}]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(image.convert('RGB'), input_text, add_special_tokens=False,
                       return_tensors='pt').to('cuda')
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOK, do_sample=False, use_cache=True,
                             eos_token_id=processor.tokenizer.eos_token_id,
                             pad_token_id=processor.tokenizer.eos_token_id)
    gen = out[0][inputs['input_ids'].shape[1]:]
    return processor.tokenizer.decode(gen, skip_special_tokens=True).strip()


## BƯỚC 4: Nạp 2 test set

In [ ]:
import io
from PIL import Image
from datasets import load_dataset

print('Tai VQA-RAD test...')
vqarad = [dict(x) for x in load_dataset('flaviagiammarino/vqa-rad')['test']]
print('Tai RadImageNet parquet...')
rad = [dict(x) for x in load_dataset('parquet', data_files=PARQUET, split='train')]
for _r in rad:
    im = _r.get('image')
    if isinstance(im, dict) and im.get('bytes'):
        _r['image'] = Image.open(io.BytesIO(im['bytes']))
if EVAL_LIMIT:
    vqarad, rad = vqarad[:EVAL_LIMIT], rad[:EVAL_LIMIT]
TESTS = {'vqarad': vqarad, 'radimagenet': rad}
print('VQA-RAD:', len(vqarad), '| RadImageNet:', len(rad))

## BƯỚC 5: EVAL 2 test set (mỗi model load 1 lần) + push HF

In [ ]:
import os, json, gc, time, torch
from tqdm.auto import tqdm
from src.train.metrics import score_predictions
from src.data.vqa_rad import normalize_answer
from src.integrations import hf_push
HF_TOKEN = os.environ.get('HF_TOKEN')

# Dọn dẹp triệt để GPU nếu có model cũ từ cell train/eval bị lỗi trước đó
for var in ['model', 'processor', 'trainer']:
    if var in globals():
        print(f"Dọn dẹp biến '{var}' cũ...")
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

summary = {}          # 'model/tag' -> metrics
vqarad_hyps = {}      # model_name -> list hyp tren vqarad (de lam CSV 50)

for m in MODELS:
    model, processor = load_gemma(m)
    # params -> DU COT nhu A1/A2: tong param + so param LoRA (trainable luc finetune)
    p_total = sum(p.numel() for p in model.parameters())
    p_train = sum(p.numel() for n, p in model.named_parameters() if 'lora' in n.lower())
    for tag, recs in TESTS.items():
        t0 = time.time()
        hyps = [gemma_predict(model, processor, r['image'], r['question'])
                for r in tqdm(recs, desc=m['name'] + '/' + tag)]
        refs = [normalize_answer(r['answer']) for r in recs]
        mm = score_predictions(refs, hyps)   # p_yes=None -> tu suy tu text, van du cot clin_*
        mm['params_total'] = p_total
        mm['params_trainable'] = p_train
        mm['inference_time_sec'] = round(time.time() - t0, 1)
        summary[m['name'] + '/' + tag] = mm
        if tag == 'vqarad':
            vqarad_hyps[m['name']] = hyps
        # luu + push metrics vao chinh repo cua model (subfolder eval/)
        os.makedirs('gemma_eval', exist_ok=True)
        fp = 'gemma_eval/' + m['name'] + '_' + tag + '_metrics.json'
        with open(fp, 'w') as f: json.dump(mm, f, indent=2)
        if PUSH_HF and HF_TOKEN:
            hf_push.push_files(m['repo'], [(fp, 'eval/' + tag + '_metrics.json')], HF_TOKEN)
        print('  ', tag, '| em', round(mm['em_overall'], 4),
              '| f1_open', round(mm.get('token_f1_open', 0), 4), '| sem_open', mm.get('sem_open'))
    del model, processor; gc.collect(); torch.cuda.empty_cache()
    print('== xong', m['name'], '==')

# Kiem tra DU COT (phai trung bo cot voi metrics.json cua A1/A2...)
print('\nCac cot metrics:', list(next(iter(summary.values())).keys()))

print('\n=== BANG TONG HOP ===')
print('run/test               em_overall  em_closed  em_open  f1_open  sem_open')
for k, mm in summary.items():
    print(k.ljust(22),
          str(round(mm['em_overall'], 4)).ljust(11),
          str(round(mm['em_closed'], 4)).ljust(10),
          str(round(mm['em_open'], 4)).ljust(8),
          str(round(mm.get('token_f1_open', 0), 4)).ljust(8),
          mm.get('sem_open'))


## BƯỚC 6: Inference CSV 50 mẫu mỗi model (soi tại sao sai)

In [ ]:
import pandas as pd, hashlib, base64
from io import BytesIO

def image_to_html(img):
    buffered = BytesIO()
    img_resized = img.resize((100, 100))
    img_resized.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return f'<img src="data:image/png;base64,{img_str}" width="100" height="100" />'

# Tạo dataframe gộp chứa ảnh để in ra
rows = {
    'image':    [image_to_html(img) for img in sample_images],
    'img_id':   [hashlib.md5(img.tobytes()).hexdigest()[:8] for img in sample_images],
    'question': sample_questions,
    'type':     sample_types,
    'gold':     sample_golds,
}

for m in MODELS:
    name = m['name']
    if name in all_preds:
        preds = all_preds[name]
        rows['pred_' + name] = preds
        rows['ok_' + name] = [int(normalize_answer(p) == normalize_answer(g)) for p, g in zip(preds, sample_golds)]

df = pd.DataFrame(rows)

# Tách ra DataFrame riêng cho từng model để lưu CSV riêng biệt (không chứa cột image)
import os
os.makedirs('gemma_eval', exist_ok=True)

total_samples = len(selected)
for m in MODELS:
    name = m['name']
    if name in all_preds:
        # Lọc ra các cột liên quan đến model này
        cols_to_keep = ['img_id', 'question', 'type', 'gold', 'pred_' + name, 'ok_' + name]
        df_single = df[cols_to_keep]
        csv_path = f'gemma_eval/inference_{total_samples}_balanced_{name}.csv'
        df_single.to_csv(csv_path, index=False)
        print(f'Saved {csv_path} shape={df_single.shape}')

print()
print('Type distribution:')
print(df['type'].value_counts())
print()

# Accuracy nhanh
for m in MODELS:
    name = m['name']
    if name in all_preds:
        ok_col = 'ok_' + name
        total_acc = df[ok_col].mean()
        open_acc  = df[df['type']=='open'][ok_col].mean()
        close_acc = df[df['type']=='closed'][ok_col].mean()
        print(f"{name:12s}  overall={total_acc:.2%}  open={open_acc:.2%}  closed={close_acc:.2%}")
